# 00 — Full pipeline

Master orchestration notebook. Runs every stage end-to-end for each well in the loaded dataset: `preprocessing → sorting → auto_merge → analyzer → auto_curation → burst_detection → iterative_burst_detection`, then renders the plate viewer once per recording. The cache makes re-runs safe — completed wells are skipped, and a config change to any task in `pipeline_config.json` (editable from the dashboard's Settings page) invalidates that task and its downstream dependents for affected wells.

## Imports

`yuxin_mea` is an installable package (`pip install -e .` once). No `sys.path` hacks needed; imports work whether you launch Jupyter from the repo root or from `notebooks/v2/`.

In [ ]:
import logging
import time
import traceback
from pathlib import Path

import pandas as pd

from yuxin_mea.config import ConfigManager
from yuxin_mea.dataset import DatasetManager
from yuxin_mea.pipeline import PipelineManager, WorkItem
from yuxin_mea.pipeline.task_record import TaskStatus
from yuxin_mea.tasks import (
    AnalyzerTask,
    AutoCurationTask,
    AutoMergeTask,
    BurstDetectionTask,
    IterativeBurstDetectionTask,
    PlateViewerTask,
    PreprocessingTask,
    SortingTask,
)

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

## Task chain

Inspect the registered task order and dependency graph before doing anything else. Well-level tasks run once per (recording, well); `PlateViewerTask` runs once per recording after all its wells have completed `burst_detection`.

In [ ]:
WELL_TASK_CLASSES = [
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
    IterativeBurstDetectionTask,
]
PLATE_TASK_CLASSES = [PlateViewerTask]
ALL_TASK_CLASSES = WELL_TASK_CLASSES + PLATE_TASK_CLASSES

TASK_ORDER = [cls.task_name for cls in WELL_TASK_CLASSES]

print("Pipeline execution order:")
for i, cls in enumerate(ALL_TASK_CLASSES, 1):
    deps = " → ".join(cls.dependencies) if cls.dependencies else "(none)"
    print(f"  {i}. {cls.task_name:<26s}  depends on: {deps}")

## Config

On the **first run** this cell writes a template and stops — edit the file (or use the dashboard's Settings page) then re-run.

Key parameters to review:
- `global.data_root` — sample/root folder containing date directories, e.g. `/path/to/Sadegh_Lab/CX138`
- `global.analysis_root` — where every task writes its outputs
- `tasks.sorting.sorter` — sorter name (default `kilosort4`)
- `tasks.auto_merge.enabled` — set `true` to merge over-split units
- `tasks.auto_curation.enabled` and `*_min` / `*_max` thresholds
- `tasks.burst_detection` and `tasks.iterative_burst_detection` parameters

In [ ]:
CONFIG_FILE = Path("../pipeline_config.json").resolve()

cm = ConfigManager()
# Path override (uncomment and edit, or set via dashboard Settings page):
# cm.set_global("data_root", "/path/to/raw/data")
# cm.set_global("analysis_root", "/path/to/analysis")

for cls in ALL_TASK_CLASSES:
    cm.register_task(cls)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it (or use `yuxin-mea-dashboard --config <path>` Settings page), then re-run."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded from: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print()
for cls in ALL_TASK_CLASSES:
    params = cm.get_task_params(cls.task_name)
    print(f"  [{cls.task_name}]  {params}")

## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_SCAN_TYPE = "Network"
TARGET_RUN_ID    = "000029"

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_recording_by([
    ("scan_type", "==", TARGET_SCAN_TYPE),
    ("run_id",    "==", TARGET_RUN_ID),
])

if not recordings:
    raise RuntimeError(
        "No recordings matched the notebook filters.\n"
        f"  data_root: {DATA_ROOT}\n"
        f"  filters: scan_type == {TARGET_SCAN_TYPE!r}, run_id == {TARGET_RUN_ID!r}\n"
        "Expected data_root to be a sample/root folder containing date directories, "
        "for example: .../Sadegh_Lab/CX138."
    )

print(f"Found {len(recordings)} {TARGET_SCAN_TYPE} recording(s)")

pd.DataFrame([
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "scan_type":     r.scan_type,
        "run_id":        r.run_id,
        "n_wells":       len(r.wells),
        "size_GB":       round(r.file_size / 1e9, 2),
    }
    for r in recordings
])

## 2. Register tasks and add wells

In [ ]:
pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

for cls in WELL_TASK_CLASSES:
    try:
        pipeline_mgr.register_task(cls)
        print(f"Registered task: {cls.task_name!r}")
    except ValueError as e:
        print(f"Task already registered ({e}) — continuing")

n_added   = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Recordings skipped: {n_skipped}")

## 3. Pipeline status overview

Matrix of every (well × task) status. The dashboard's `/pipeline` page renders this same matrix interactively — use whichever you prefer.

In [ ]:
_STATUS_SYMBOL = {
    str(TaskStatus.NOT_RUN):  "—",
    str(TaskStatus.RUNNING):  "⏳",
    str(TaskStatus.COMPLETE): "✓",
    str(TaskStatus.FAILED):   "✗",
}


def _status_table(mgr: PipelineManager) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        if "/" not in entry.well_id:
            continue
        rec_name, well_id = entry.well_id.split("/", 1)
        row: dict = {
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
        }
        for task_name in TASK_ORDER:
            t = entry.tasks.get(task_name)
            status_str = str(t.status) if t else str(TaskStatus.NOT_RUN)
            row[task_name] = _STATUS_SYMBOL.get(status_str, status_str)
        rows.append(row)
    columns = ["recording_key", "rec_name", "well_id", *TASK_ORDER]
    return pd.DataFrame(rows, columns=columns)


df_status = _status_table(pipeline_mgr)
print("Legend:  — not run   ⏳ running   ✓ complete   ✗ failed")
df_status

## 4. Refresh controls (optional)

Use before the main loop to re-run selected work. `PipelineManager.refresh()` applies to cached per-well tasks. `plate_viewer` runs once per recording outside the per-well cache, so refreshing it means regenerating the HTML output.

In [ ]:
# Optional per-well task refresh examples. Uncomment one before running the pipeline.

# One task across all wells, including downstream dependents:
# pipeline_mgr.refresh("sorting")

# One task for one recording, including downstream dependents:
# pipeline_mgr.refresh(
#     "sorting",
#     recording_key="SampleA/240415/PlateX/Network/001",
# )

# One task for one well, including downstream dependents:
# pipeline_mgr.refresh(
#     "sorting",
#     recording_key="SampleA/240415/PlateX/Network/001",
#     well_id="rec0000/well000",
# )

# Plate viewer refresh. Skips well-level tasks below and regenerates plate_viewer.html
# for selected recordings whose burst_detection is complete.
REFRESH_PLATE_VIEWER_ONLY = False

# Empty set means all currently loaded recordings.
PLATE_VIEWER_RECORDING_KEYS: set[str] = set()
# PLATE_VIEWER_RECORDING_KEYS = {"SampleA/240415/PlateX/Network/001"}

## 5. Run full pipeline

`PipelineManager.get_next_task()` respects dependency order — it only returns a task once all its upstream tasks are complete for that well. The loop below drains well-level tasks first, then runs `PlateViewerTask` once per recording whose loaded wells have completed `burst_detection`.

**Knobs:**
- `RETRY_FAILED_TASKS` — set of task names to re-poll if they previously failed.
- `STOP_AFTER_TASK` — task name to pause after (skips plate viewer).

In [ ]:
# ── Control knobs ──────────────────────────────────────────────────────────────
RETRY_FAILED_TASKS: set[str] = set()      # e.g. {"analyzer", "auto_curation"}
STOP_AFTER_TASK:    str | None = None     # e.g. "sorting" to pause after sorting
_refresh_plate_viewer_only = bool(globals().get("REFRESH_PLATE_VIEWER_ONLY", False))
# ──────────────────────────────────────────────────────────────────────────────

_task_instances: dict[str, object] = {cls.task_name: cls() for cls in WELL_TASK_CLASSES}
_rec_lookup:     dict[str, object] = {r.cache_key: r for r in recordings}
_completed_stages: list[str]       = []

# Scope get_next_task() to the recordings loaded in this session so stale cache
# entries from previous runs aren't picked up.
_current_recording_keys = set(_rec_lookup.keys())

if _refresh_plate_viewer_only:
    print("\nSkipping well-level tasks; refreshing plate viewer only.")

while not _refresh_plate_viewer_only:
    work_items = pipeline_mgr.get_next_task(
        n=1, retry_failed=False, recording_keys=_current_recording_keys
    )

    # Also poll failed tasks that are explicitly marked for retry.
    if not work_items and RETRY_FAILED_TASKS:
        work_items = [
            item
            for item in pipeline_mgr.get_next_task(
                n=1, retry_failed=True, recording_keys=_current_recording_keys
            )
            if item.task_name in RETRY_FAILED_TASKS
        ]

    if not work_items:
        break

    item:      WorkItem = work_items[0]
    task                = _task_instances[item.task_name]
    rec_entry           = _rec_lookup[item.recording_key]
    rec_name, well_id   = item.well_id.split("/", 1)
    params              = cm.get_task_params(item.task_name)

    print(f"\n[{item.task_name}]  {item.recording_key} / {rec_name} / {well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)

    t0 = time.perf_counter()
    try:
        output_path = task.run(
            item.recording_key,
            item.well_id,
            dataset_mgr.get_path(rec_entry),
            params,
        )
        elapsed = time.perf_counter() - t0
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output_path)
        print(f"  ✓  {elapsed:.1f}s  → {output_path}")
        _completed_stages.append(item.task_name)
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  ✗  {elapsed:.1f}s  failed: {exc}")

    if STOP_AFTER_TASK and item.task_name == STOP_AFTER_TASK:
        print(f"\nPaused after {STOP_AFTER_TASK!r} as requested.")
        break

if not _refresh_plate_viewer_only:
    print("\n— No more pending well-level tasks. —")


def _loaded_well_entries(recording_key: str):
    return [
        entry
        for entry in pipeline_mgr.get_entries_for_recording(recording_key)
        if "/" in entry.well_id
    ]


def _burst_detection_complete_for_loaded_wells(recording_key: str) -> bool:
    entries = _loaded_well_entries(recording_key)
    if not entries:
        return False
    return all(
        (task := entry.tasks.get(BurstDetectionTask.task_name)) is not None
        and task.status == TaskStatus.COMPLETE
        for entry in entries
    )


plate_viewer = PlateViewerTask()
plate_params = cm.get_task_params(PlateViewerTask.task_name)
plate_outputs: dict[str, Path] = {}

_requested_plate_recording_keys = set(
    globals().get("PLATE_VIEWER_RECORDING_KEYS", set()) or _current_recording_keys
)
_plate_recording_keys = _requested_plate_recording_keys & _current_recording_keys
_missing_plate_recording_keys = _requested_plate_recording_keys - _current_recording_keys
for recording_key in sorted(_missing_plate_recording_keys):
    print(f"\n[plate_viewer]  {recording_key}  skipped: not loaded in this session")

if STOP_AFTER_TASK is None or _refresh_plate_viewer_only:
    for recording_key in sorted(_plate_recording_keys):
        rec_entry = _rec_lookup[recording_key]
        if not _burst_detection_complete_for_loaded_wells(recording_key):
            print(f"\n[plate_viewer]  {recording_key}  skipped: burst_detection incomplete")
            continue

        print(f"\n[plate_viewer]  {recording_key}")
        t0 = time.perf_counter()
        try:
            output_path = plate_viewer.run(
                recording_key,
                "__plate__",
                dataset_mgr.get_path(rec_entry),
                plate_params,
            )
            elapsed = time.perf_counter() - t0
            plate_outputs[recording_key] = output_path
            print(f"  ✓  {elapsed:.1f}s  → {output_path}")
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            traceback.print_exc()
            print(f"  ✗  {elapsed:.1f}s  failed: {exc}")
else:
    print("\nPlate viewer skipped because STOP_AFTER_TASK is set.")

## 6. Final status report

In [ ]:
df_final = _status_table(pipeline_mgr)

if df_final.empty:
    print("No pipeline entries are registered for the current session.")
else:
    print("=== Per-task summary ===")
    for task_name in TASK_ORDER:
        col = df_final[task_name]
        counts = col.value_counts().to_dict()
        print(f"  {task_name:<26s}  {counts}")

failed_mask = (
    (df_final[TASK_ORDER] == "✗").any(axis=1)
    if not df_final.empty
    else pd.Series(dtype=bool)
)
if not df_final.empty and failed_mask.any():
    print("\n=== Failed wells ===")
    for _, row in df_final[failed_mask].iterrows():
        failed_tasks = [t for t in TASK_ORDER if row[t] == "✗"]
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    failed stages: {failed_tasks}")

if globals().get("plate_outputs"):
    print("\n=== Plate viewer outputs ===")
    for recording_key, output_path in plate_outputs.items():
        print(f"  {recording_key}: {output_path}")

print("\nLegend:  — not run   ⏳ running   ✓ complete   ✗ failed")
df_final

## 7. Curation summary

Reads `quality_metrics.pkl` from every completed `auto_curation` well and produces an across-wells summary table.

In [ ]:
summary_rows = []

for entry in pipeline_mgr.entries:
    if "/" not in entry.well_id:
        continue
    curation_task = entry.tasks.get(AutoCurationTask.task_name)
    if (
        not curation_task
        or curation_task.status != TaskStatus.COMPLETE
        or not curation_task.output_path
    ):
        continue

    qm_path = Path(curation_task.output_path) / "quality_metrics.pkl"
    if not qm_path.exists():
        continue

    rec_name, well_id = entry.well_id.split("/", 1)
    qm = pd.read_pickle(qm_path)

    curated = qm[qm["curated"]]
    summary_rows.append({
        "recording_key": entry.recording_key,
        "rec_name":      rec_name,
        "well_id":       well_id,
        "n_total":        len(qm),
        "n_curated":      len(curated),
        "pct_kept":       round(100 * len(curated) / len(qm), 1) if len(qm) else 0,
        "median_fr_hz":   round(curated["firing_rate"].median(), 3)
                          if "firing_rate" in curated.columns and len(curated) else None,
        "median_amp_uv":  round(curated["amplitude_median"].median(), 1)
                          if "amplitude_median" in curated.columns and len(curated) else None,
    })

if summary_rows:
    df_curation = pd.DataFrame(summary_rows)
    print(f"Wells with curation results: {len(df_curation)}")
    print(f"Total curated units: {df_curation['n_curated'].sum()}  /  {df_curation['n_total'].sum()}")
    df_curation
else:
    print("No curation results yet — run the pipeline first.")